In [1]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np

df = pd.read_csv('preprocess_1.csv')


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 578 entries, 0 to 577
Data columns (total 35 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      578 non-null    int64  
 1   Name            578 non-null    str    
 2   Screen Size     578 non-null    float64
 3   Display         578 non-null    str    
 4   NFC             578 non-null    int64  
 5   Battery         578 non-null    float64
 6   antutu_11       578 non-null    float64
 7   clock           578 non-null    float64
 8   Brand           578 non-null    str    
 9   OS_Name         578 non-null    str    
 10  OS_Version      578 non-null    float64
 11  Refresh Rate    578 non-null    float64
 12  total_cores     578 non-null    float64
 13  max_freq_ghz    578 non-null    float64
 14  min_freq_ghz    578 non-null    float64
 15  weighted_mean   578 non-null    float64
 16  rear_count      578 non-null    float64
 17  rear_mp_max     578 non-null    float64
 18  r

In [3]:
df.drop(columns=['clock', 'RAM_max', 'ROM_max', 'weighted_mean', 'max_freq_ghz', 'gpu_gen', 'Unnamed: 0'], inplace=True)

Fix OS_version

In [4]:
df.loc[df['OS_Version'] == 26.0, 'OS_Version'] = 19.0

print("OS_Version fix applied:")
print(df[df['Name'].str.contains('iphone 17|iphone air', case=False, na=False)][
    ['Name', 'OS_Version']
].to_string())
print()

OS_Version fix applied:
                  Name  OS_Version
125          iphone 17        19.0
126      iphone 17 pro        19.0
127  iphone 17 pro max        19.0
128         iphone 17e        19.0
129         iphone air        19.0



Fix MediaTekGPU

In [5]:
powervr_gens = ['G35', 'G36', 'P35', 'P95', 'A22', '930']
mediatek_powervr_mask = (
    (df['Chipset_name'] == 'MediaTek') &
    (df['Chipset_gen'].isin(powervr_gens))
)
df.loc[mediatek_powervr_mask, 'gpu_name'] = 'PowerVR'

# 'Other' GPU on MediaTek — Dimensity 7020 and 7200 use Mali
mediatek_other_mask = (
    (df['Chipset_name'] == 'MediaTek') &
    (df['gpu_name'] == 'Other')
)
df.loc[mediatek_other_mask, 'gpu_name'] = 'Mali'

print("MediaTek GPU fix applied:")
print(df[df['Chipset_name'] == 'MediaTek']['gpu_name'].value_counts())
print()


MediaTek GPU fix applied:
gpu_name
Mali       133
PowerVR     14
Name: count, dtype: int64



3. Rare Label Bucketing

In [6]:
# Chipset_gen — collapse labels appearing < 3 times into 'Other'
chipset_gen_freq = df['Chipset_gen'].value_counts()
rare_chipset_gen = chipset_gen_freq[chipset_gen_freq < 3].index
df['Chipset_gen'] = df['Chipset_gen'].apply(
    lambda x: 'Other' if x in rare_chipset_gen else x
)

# Brand — collapse brands with < 5 phones into 'Other'
brand_freq = df['Brand'].value_counts()
rare_brands = brand_freq[brand_freq < 5].index
df['Brand'] = df['Brand'].apply(
    lambda x: 'Other' if x in rare_brands else x
)

# Verification
print("=== Chipset_gen after bucketing ===")
print(f"Unique labels: {df['Chipset_gen'].nunique()}")
print(f"Remaining rare (< 3): {(df['Chipset_gen'].value_counts() < 3).sum()}")
print()

print("=== Brand after bucketing ===")
print(df['Brand'].value_counts())
print()
print(f"Remaining rare (< 5): {(df['Brand'].value_counts() < 5).sum()}")

=== Chipset_gen after bucketing ===
Unique labels: 61
Remaining rare (< 3): 0

=== Brand after bucketing ===
Brand
samsung     82
xiaomi      81
vivo        75
oppo        66
honor       42
realme      31
Other       28
huawei      28
nubia       27
apple       25
tecno       21
google      20
oneplus     15
meizu        7
motorola     7
infinix      6
nokia        6
sony         6
asus         5
Name: count, dtype: int64

Remaining rare (< 5): 0


4. Need Tags

In [7]:
df_tags = df.copy()

df_tags['Gaming_Need']        = ((df_tags['antutu_11']    >  900_000) &
                                  (df_tags['Refresh Rate'] >= 120)).astype(int)

df_tags['Battery_Need']       = (df_tags['Battery']        >= 5500).astype(int)

df_tags['Photography_Need']   = ((df_tags['rear_ois']      == 1) &
                                  (df_tags['rear_count']    >= 2)).astype(int)

df_tags['Performance_Need']   = (df_tags['antutu_11']      > 1_200_000).astype(int)

df_tags['Large_Display_Need'] = (df_tags['Screen Size']    >= 6.8).astype(int)

df_tags['HighRes_Need']       = (df_tags['PPI']            >= 400).astype(int)

df_tags['Multitask_Need']     = (df_tags['RAM_min']        >= 12).astype(int)

NEED_TAGS = [c for c in df_tags.columns if c.endswith('_Need')]
print("=== Need Tag Distribution ===")
print(df_tags[NEED_TAGS].sum().sort_values(ascending=False))
print()

=== Need Tag Distribution ===
Gaming_Need           221
HighRes_Need          189
Performance_Need      183
Multitask_Need        150
Battery_Need           98
Large_Display_Need     76
Photography_Need       64
dtype: int64



5. Log Transform (np.log1p)

In [8]:
LOG_COLS = ['antutu_11', 'Battery', 'Screen Size', 'rear_mp_max']

for col in LOG_COLS:
    df[col] = np.log1p(df[col])

print("Log transform applied to:", LOG_COLS)
print(df[LOG_COLS].skew().round(3))
print()


Log transform applied to: ['antutu_11', 'Battery', 'Screen Size', 'rear_mp_max']
antutu_11      0.217
Battery       -0.650
Screen Size   -0.122
rear_mp_max   -1.172
dtype: float64



6. MinMaxScale

In [9]:
BINARY_COLS   = ['NFC', 'has_eSIM', 'rear_ois', 'rear_telephoto', 'rear_wide']
EXCLUDE_COLS  = BINARY_COLS + ['Name']

SCALE_COLS = [
    c for c in df.select_dtypes(include=np.number).columns
    if c not in EXCLUDE_COLS
]


print("=== Columns Being Scaled ===")
print(SCALE_COLS)
print()

scaler = MinMaxScaler()
df[SCALE_COLS] = scaler.fit_transform(df[SCALE_COLS])

=== Columns Being Scaled ===
['Screen Size', 'Battery', 'antutu_11', 'OS_Version', 'Refresh Rate', 'total_cores', 'min_freq_ghz', 'rear_count', 'rear_mp_max', 'rear_f/', 'front_mp', 'front_f/', 'PPI', 'SIM_total', 'RAM_min', 'ROM_min']



In [10]:


print("=== Scaling Complete — Verification (min=0, max=1) ===")
print(df[SCALE_COLS].agg(['min', 'max']).round(3))

=== Scaling Complete — Verification (min=0, max=1) ===
     Screen Size  Battery  antutu_11  OS_Version  Refresh Rate  total_cores  \
min          0.0      0.0        0.0         0.0           0.0          0.0   
max          1.0      1.0        1.0         1.0           1.0          1.0   

     min_freq_ghz  rear_count  rear_mp_max  rear_f/  front_mp  front_f/  PPI  \
min           0.0         0.0          0.0      0.0       0.0       0.0  0.0   
max           1.0         1.0          1.0      1.0       1.0       1.0  1.0   

     SIM_total  RAM_min  ROM_min  
min        0.0      0.0      0.0  
max        1.0      1.0      1.0  


8. Encoding

In [11]:
# --- 6a. Binary encode OS_Name (only 2 values) ---
df['is_iOS'] = (df['OS_Name'] == 'iOS').astype(int)
df.drop(columns=['OS_Name'], inplace=True)

# --- 6b. One-hot encode low-cardinality categorical columns ---
OHE_COLS = ['Display', 'Chipset_name', 'gpu_name']

df = pd.get_dummies(df, columns=OHE_COLS, prefix=OHE_COLS, dtype=int)

# --- 6c. Frequency encode Brand ---
# Brand has 19 unique values after bucketing — OHE would add too many columns
# Frequency encoding replaces brand with its normalised market share (0-1)
brand_freq_map = df['Brand'].value_counts(normalize=True)
df['Brand_freq'] = df['Brand'].map(brand_freq_map)
df.drop(columns=['Brand'], inplace=True)

# --- 6d. Drop Chipset_gen (still too many labels even after bucketing) ---
df.drop(columns=['Chipset_gen'], inplace=True)

# --- 6e. Drop Name (identifier, not a feature) ---
phone_names = df['Name'].copy()
df.drop(columns=['Name'], inplace=True)

# Verification
print("=== Encoding Complete ===")
print(f"Final shape: {df.shape}")
print()
print("=== New columns added ===")
print([c for c in df.columns if '_' in c][-20:])
print()
print("=== Remaining object columns (should be none) ===")
print(df.select_dtypes(include='object').columns.tolist())

=== Encoding Complete ===
Final shape: (578, 42)

=== New columns added ===
['Display_AMOLED', 'Display_IPS LCD', 'Display_LCD', 'Display_OLED', 'Display_Other', 'Chipset_name_Apple', 'Chipset_name_Exynos', 'Chipset_name_Google', 'Chipset_name_Kirin', 'Chipset_name_MediaTek', 'Chipset_name_Other', 'Chipset_name_Snapdragon', 'Chipset_name_Unisoc', 'gpu_name_Adreno', 'gpu_name_Apple GPU', 'gpu_name_Maleoon', 'gpu_name_Mali', 'gpu_name_PowerVR', 'gpu_name_Xclipse', 'Brand_freq']

=== Remaining object columns (should be none) ===
[]


8. Save Output

In [ ]:
# --- 8a. Readable version (unscaled + Name + need tags) ---
readable_cols = ['Name'] + NEED_TAGS + [
    'Brand', 'antutu_11', 'Battery', 'Screen Size', 'RAM_min', 'ROM_min',
    'Refresh Rate', 'rear_mp_max', 'PPI', 'rear_ois', 'rear_count',
    'rear_telephoto', 'rear_wide', 'NFC', 'has_eSIM', 'OS_Name'
]
df_readable = df_tags[readable_cols].copy()
df_readable.to_csv('preprocess_2_readable.csv', index=False)
print(f"Readable version saved — shape: {df_readable.shape}")

# --- 8b. Apply log transform + MinMaxScaler to df_tags ---
for col in LOG_COLS:
    df_tags[col] = np.log1p(df_tags[col])

df_tags[SCALE_COLS] = scaler.fit_transform(df_tags[SCALE_COLS])

# --- 8c. Apply encoding ---
df_tags['is_iOS'] = (df_tags['OS_Name'] == 'iOS').astype(int)
df_tags.drop(columns=['OS_Name'], inplace=True)

brand_freq_map = df_tags['Brand'].value_counts(normalize=True)
df_tags['Brand_freq'] = df_tags['Brand'].map(brand_freq_map)
df_tags.drop(columns=['Brand'], inplace=True)

df_tags = pd.get_dummies(df_tags, columns=OHE_COLS, prefix=OHE_COLS, dtype=int)
df_tags.drop(columns=['Chipset_gen'], inplace=True)

# --- 8d. Build and Save Final Matrices ---     
phone_names    = df_tags['Name'].copy()
tag_matrix     = pd.concat([phone_names, df_tags[NEED_TAGS]], axis=1)
feature_matrix = df_tags.drop(columns=['Name'] + NEED_TAGS)

print("=== Feature Matrix ===")
print(f"Shape: {feature_matrix.shape}")
print(f"Columns ({len(feature_matrix.columns)}):")
print(feature_matrix.columns.tolist())
print()
print("=== Tag Matrix ===")
print(f"Shape: {tag_matrix.shape}")
print(tag_matrix[NEED_TAGS].sum().sort_values(ascending=False))
print()
print("=== Null check ===")
print(f"Feature matrix nulls: {feature_matrix.isna().sum().sum()}")
print(f"Tag matrix nulls:     {tag_matrix.isna().sum().sum()}")
print()
print("=== Value range (all should be 0-1) ===")
print(feature_matrix.agg(['min', 'max']).round(3))

feature_matrix.to_csv('feature_matrix.csv', index=False)
tag_matrix.to_csv('tag_matrix.csv',         index=False)
phone_names.to_csv('phone_names.csv',        index=False)

print()
print(f"feature_matrix.csv → {feature_matrix.shape}")
print(f"tag_matrix.csv     → {tag_matrix.shape}")
print(f"phone_names.csv    → {len(phone_names)} entries")

Readable version saved — shape: (578, 24)
=== Feature Matrix ===
Shape: (578, 42)
Columns (42):
['Screen Size', 'NFC', 'Battery', 'antutu_11', 'OS_Version', 'Refresh Rate', 'total_cores', 'min_freq_ghz', 'rear_count', 'rear_mp_max', 'rear_f/', 'rear_ois', 'rear_telephoto', 'rear_wide', 'front_mp', 'front_f/', 'PPI', 'SIM_total', 'has_eSIM', 'RAM_min', 'ROM_min', 'is_iOS', 'Brand_freq', 'Display_AMOLED', 'Display_IPS LCD', 'Display_LCD', 'Display_OLED', 'Display_Other', 'Chipset_name_Apple', 'Chipset_name_Exynos', 'Chipset_name_Google', 'Chipset_name_Kirin', 'Chipset_name_MediaTek', 'Chipset_name_Other', 'Chipset_name_Snapdragon', 'Chipset_name_Unisoc', 'gpu_name_Adreno', 'gpu_name_Apple GPU', 'gpu_name_Maleoon', 'gpu_name_Mali', 'gpu_name_PowerVR', 'gpu_name_Xclipse']

=== Tag Matrix ===
Shape: (578, 8)
Gaming_Need           221
HighRes_Need          189
Performance_Need      183
Multitask_Need        150
Battery_Need           98
Large_Display_Need     76
Photography_Need       64
dty

In [15]:
fm = pd.read_csv('feature_matrix.csv')
tm = pd.read_csv('tag_matrix.csv')
pn = pd.read_csv('phone_names.csv')
rd = pd.read_csv('preprocess_2_readable.csv')

# All should match
assert list(pn['Name']) == list(tm['Name']), "Name mismatch: phone_names vs tag_matrix"
assert list(pn['Name']) == list(rd['Name']), "Name mismatch: phone_names vs readable"
assert len(fm) == len(tm) == len(pn) == len(rd), "Row count mismatch"

print("All files aligned correctly")

All files aligned correctly


In [14]:
# # ── Build Final Matrices ──────────────────────────────────────────────────────

# # --- Feature matrix (for distance calculation) ---
# # Excludes: Name, need tags, pipeline flags
# EXCLUDE_FROM_MATRIX = ['Name'] + NEED_TAGS

# feature_matrix = df_model.drop(
#     columns=[c for c in EXCLUDE_FROM_MATRIX if c in df_model.columns]
# )

# # --- Need tag matrix (for filtering before distance calculation) ---
# tag_matrix = df_tags[['Name'] + NEED_TAGS].copy()

# # --- Verification ---
# print("=== Feature Matrix ===")
# print(f"Shape: {feature_matrix.shape}")
# print(f"Columns ({len(feature_matrix.columns)}):")
# print(feature_matrix.columns.tolist())
# print()
# print("=== Tag Matrix ===")
# print(f"Shape: {tag_matrix.shape}")
# print(tag_matrix[NEED_TAGS].sum().sort_values(ascending=False))
# print()

# # --- Confirm no nulls and correct value range ---
# print("=== Null check ===")
# print(f"Feature matrix nulls: {feature_matrix.isna().sum().sum()}")
# print(f"Tag matrix nulls: {tag_matrix.isna().sum().sum()}")
# print()
# print("=== Value range check (all should be 0-1 or binary) ===")
# print(feature_matrix.agg(['min', 'max']).round(3))

# # --- Save ---
# feature_matrix.to_csv('feature_matrix.csv', index=False)
# tag_matrix.to_csv('tag_matrix.csv', index=False)
# phone_names.to_csv('phone_names.csv', index=False)

# print()
# print(f"feature_matrix.csv  → {feature_matrix.shape}")
# print(f"tag_matrix.csv      → {tag_matrix.shape}")
# print(f"phone_names.csv     → {len(phone_names)} entries")